In [1]:
# ============================================================
# GENERATIVE AI PRACTICAL
# Fine-Tuning a Causal Language Model using Hugging Face
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL REQUIRED LIBRARIES
# ------------------------------------------------------------

%pip install -q torch transformers datasets accelerate safetensors


# ------------------------------------------------------------
# 2. IMPORT LIBRARIES
# ------------------------------------------------------------

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

print("Libraries imported successfully!")


# ------------------------------------------------------------
# 3. CHECK DEVICE
# ------------------------------------------------------------

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)


# ------------------------------------------------------------
# 4. SELECT MODEL
# ------------------------------------------------------------

# GPT-2 is small and suitable for a practical/demo.
# It downloads automatically from Hugging Face.

model_name = "gpt2"

print("Loading tokenizer...")


# ------------------------------------------------------------
# 5. LOAD TOKENIZER
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT-2 does not have a padding token by default.
# Use EOS token as padding token.

tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully!")


# ------------------------------------------------------------
# 6. LOAD PRE-TRAINED MODEL
# ------------------------------------------------------------

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded successfully!")


# ------------------------------------------------------------
# 7. MOVE MODEL TO DEVICE
# ------------------------------------------------------------

model = model.to(device)

print("Model moved to:", device)


# ------------------------------------------------------------
# 8. PREPARE TRAINING DATASET
# ------------------------------------------------------------

texts = [
    "Artificial intelligence is transforming the way people solve complex problems.",
    
    "Machine learning allows computers to learn patterns from data.",
    
    "Deep learning uses neural networks to process complex information.",
    
    "Generative AI can create text, images, audio, and other forms of content.",
    
    "Natural language processing enables computers to understand human language.",
    
    "Large language models are trained on large amounts of textual data.",
    
    "Transformers use attention mechanisms to understand relationships between words.",
    
    "Python is widely used for artificial intelligence and machine learning.",
    
    "Data is an important component of modern artificial intelligence systems.",
    
    "Generative artificial intelligence is becoming an important technology in software development."
]

dataset = Dataset.from_dict({
    "text": texts
})

print("Dataset created successfully!")
print("Number of training examples:", len(dataset))


# ------------------------------------------------------------
# 9. TOKENIZE DATASET
# ------------------------------------------------------------

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print("Dataset tokenized successfully!")


# ------------------------------------------------------------
# 10. DATA COLLATOR
# ------------------------------------------------------------

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator created successfully!")


# ------------------------------------------------------------
# 11. TRAINING CONFIGURATION
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    
    # Number of training epochs
    num_train_epochs=1,
    
    # Batch size
    per_device_train_batch_size=1,
    
    # Save model after training
    save_strategy="epoch",
    
    # Logging
    logging_steps=1,
    
    # Disable external reporting
    report_to="none",
    
    # Keep training simple for the practical
    learning_rate=5e-5,
    
    weight_decay=0.01
)


# ------------------------------------------------------------
# 12. CREATE TRAINER
# ------------------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Trainer created successfully!")


# ------------------------------------------------------------
# 13. TRAIN / FINE-TUNE MODEL
# ------------------------------------------------------------

print("\nStarting model training...")
print("Please wait...\n")

trainer.train()

print("\nTraining completed successfully!")


# ------------------------------------------------------------
# 14. SAVE FINE-TUNED MODEL
# ------------------------------------------------------------

output_directory = "./gpt2-finetuned"

trainer.save_model(output_directory)
tokenizer.save_pretrained(output_directory)

print("Fine-tuned model saved at:", output_directory)


# ------------------------------------------------------------
# 15. GENERATE TEXT
# ------------------------------------------------------------

print("\nGenerating text...\n")

prompt = "Artificial intelligence is"

# Tokenize prompt
inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

# Move input to same device as model
inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

# Generate text
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )


# ------------------------------------------------------------
# 16. DECODE GENERATED TEXT
# ------------------------------------------------------------

generated_text = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("========================================")
print("GENERATED TEXT")
print("========================================")
print(generated_text)
print("========================================")


# ------------------------------------------------------------
# 17. TEST WITH ANOTHER PROMPT
# ------------------------------------------------------------

prompt2 = "Machine learning is"

inputs2 = tokenizer(
    prompt2,
    return_tensors="pt"
)

inputs2 = {
    key: value.to(device)
    for key, value in inputs2.items()
}

with torch.no_grad():
    output2 = model.generate(
        **inputs2,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text2 = tokenizer.decode(
    output2[0],
    skip_special_tokens=True
)

print("\n========================================")
print("SECOND GENERATED TEXT")
print("========================================")
print(generated_text2)
print("========================================")


# ------------------------------------------------------------
# 18. FINAL STATUS
# ------------------------------------------------------------

print("\n========================================")
print("PRACTICAL COMPLETED SUCCESSFULLY")
print("========================================")
print("Model       :", model_name)
print("Device      :", device)
print("Dataset size:", len(dataset))
print("Output path :", output_directory)


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Libraries imported successfully!
Using device: cpu
Loading tokenizer...
Tokenizer loaded successfully!
Loading model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded successfully!
Model moved to: cpu
Dataset created successfully!
Number of training examples: 10


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Dataset tokenized successfully!
Data collator created successfully!
Trainer created successfully!

Starting model training...
Please wait...



... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed successfully!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved at: ./gpt2-finetuned

Generating text...

GENERATED TEXT
Artificial intelligence is a complex system of technology that is constantly evolving. The Internet has been a technology for many years. It is also a technology for the advancement of the human society.

The Internet has evolved in many different ways. We have made advances in technology

SECOND GENERATED TEXT
Machine learning is an area of science. It is not just about neural networks, it is about the underlying structures that enable neural networks to process information. These networks have different properties that enable them to process information.

But neural networks don't just process information.

PRACTICAL COMPLETED SUCCESSFULLY
Model       : gpt2
Device      : cpu
Dataset size: 10
Output path : ./gpt2-finetuned
